## 10.7. Sequence-to-Sequence Learning for Machine Translation

In [2]:
import collections
import math
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

### 10.7.1. Teacher Forcing

### 10.7.2. Encoder

In [3]:
def init_seq2seq(module):  #@save
    """Initialize weights for sequence-to-sequence learning."""
    if type(module) == nn.Linear:
         nn.init.xavier_uniform_(module.weight)
    if type(module) == nn.GRU:
        for param in module._flat_weights_names:
            if "weight" in param:
                nn.init.xavier_uniform_(module._parameters[param])

class Seq2SeqEncoder(d2l.Encoder):  #@save
    """The RNN encoder for sequence-to-sequence learning."""
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers,
                 dropout=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = d2l.GRU(embed_size, num_hiddens, num_layers, dropout)
        self.apply(init_seq2seq)

    def forward(self, X, *args):
        # X shape: (batch_size, num_steps)
        embs = self.embedding(X.t().type(torch.int64))
        # embs shape: (num_steps, batch_size, embed_size)
        outputs, state = self.rnn(embs)
        # outputs shape: (num_steps, batch_size, num_hiddens)
        # state shape: (num_layers, batch_size, num_hiddens)
        return outputs, state

In [4]:
vocab_size, embed_size, num_hiddens, num_layers = 10, 8, 16, 2
batch_size, num_steps = 4, 9
encoder = Seq2SeqEncoder(vocab_size, embed_size, num_hiddens, num_layers)
X = torch.zeros((batch_size, num_steps))
enc_outputs, enc_state = encoder(X)
d2l.check_shape(enc_outputs, (num_steps, batch_size, num_hiddens))

In [5]:
d2l.check_shape(enc_state, (num_layers, batch_size, num_hiddens))

In [16]:
X = torch.zeros((batch_size, num_steps))
X.shape, X.t()

(torch.Size([4, 9]),
 tensor([[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]]))

In [13]:
test_embed = nn.Embedding(vocab_size, embed_size)
test_embed(X.t().type(torch.int64)).shape, test_embed(X.t().type(torch.int64))

(torch.Size([9, 4, 8]),
 tensor([[[-1.5726,  0.9453, -1.5440,  0.1823, -0.0902, -0.2554, -0.8227,
           -0.1198],
          [-1.5726,  0.9453, -1.5440,  0.1823, -0.0902, -0.2554, -0.8227,
           -0.1198],
          [-1.5726,  0.9453, -1.5440,  0.1823, -0.0902, -0.2554, -0.8227,
           -0.1198],
          [-1.5726,  0.9453, -1.5440,  0.1823, -0.0902, -0.2554, -0.8227,
           -0.1198]],
 
         [[-1.5726,  0.9453, -1.5440,  0.1823, -0.0902, -0.2554, -0.8227,
           -0.1198],
          [-1.5726,  0.9453, -1.5440,  0.1823, -0.0902, -0.2554, -0.8227,
           -0.1198],
          [-1.5726,  0.9453, -1.5440,  0.1823, -0.0902, -0.2554, -0.8227,
           -0.1198],
          [-1.5726,  0.9453, -1.5440,  0.1823, -0.0902, -0.2554, -0.8227,
           -0.1198]],
 
         [[-1.5726,  0.9453, -1.5440,  0.1823, -0.0902, -0.2554, -0.8227,
           -0.1198],
          [-1.5726,  0.9453, -1.5440,  0.1823, -0.0902, -0.2554, -0.8227,
           -0.1198],
          [-1.5726, 